# NovaHiring — Test E2E Real (con OpenAI)

Este notebook llama al sistema **de verdad**:

1. Crea sesiones reales en la base de datos
2. Envía respuestas reales del dataset a los endpoints
3. El servidor llama a **OpenAI gpt-4o-mini** para evaluar cada respuesta
4. Calcula el score final ponderado con el Scorer
5. Muestra los scores reales vs los scores esperados

## Requisitos antes de correr
```bash
docker compose up -d          # PostgreSQL + Redis
uv run python seed.py         # Datos sembrados
uv run python seed.py use-openai  # Evaluador usando gpt-4o-mini
uv run fastapi dev main.py    # Servidor corriendo en :8000
```

**Costo estimado:** ~$0.01–0.03 por candidato (8 llamadas a gpt-4o-mini de ~500 tokens cada una).

**Tiempo estimado:** ~30–60 segundos por candidato.

In [1]:
import json
import sys
import time
from decimal import Decimal
from pathlib import Path

import httpx

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONVERSATIONS_DIR = ROOT / 'docs' / 'data' / 'conversations'
BASE_URL = 'http://localhost:8000'
JOB_ID = 'job-clinica-salud-valencia-001'

# Candidatos APTO → escenario de conversación
CANDIDATES = [
    {'candidate_id': 'cand-13-sofia-delgado',  'scenario': 'ganador_claro',       'name': 'Sofía Delgado'},
    {'candidate_id': 'cand-12-carlos-rivas',    'scenario': 'buen_candidato',      'name': 'Carlos Rivas'},
    {'candidate_id': 'cand-11-elena-martinez',  'scenario': 'candidato_promedio',  'name': 'Elena Martínez'},
]

client = httpx.Client(base_url=BASE_URL, timeout=120.0)
print('✓ Cliente HTTP listo')

✓ Cliente HTTP listo


In [2]:
# ── Paso 1: verificar que el servidor está corriendo ──────────────────────────
try:
    r = client.get('/health')
    r.raise_for_status()
    print(f'✓ Servidor OK → {r.json()}')
except httpx.ConnectError:
    print('✗ El servidor no está corriendo.')
    print('  Ejecuta: uv run fastapi dev main.py')
    raise SystemExit(1)

✓ Servidor OK → {'status': 'ok'}


In [3]:
# ── Paso 2: verificar que el proveedor activo es OpenAI ───────────────────────
import os
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

openai_key = os.getenv('OPENAI_API_KEY', '')
if not openai_key or openai_key == 'sk-...':
    print('✗ OPENAI_API_KEY no configurada en .env')
    raise SystemExit(1)

print(f'✓ OPENAI_API_KEY configurada: {openai_key[:8]}...')
print('  Para cambiar a OpenAI: uv run python seed.py use-openai')

✓ OPENAI_API_KEY configurada: sk-proj-...
  Para cambiar a OpenAI: uv run python seed.py use-openai


In [4]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_scenario(name: str) -> dict:
    return json.loads((CONVERSATIONS_DIR / f'{name}.json').read_text(encoding='utf-8'))


def get_session_state(session_id: str) -> dict:
    r = client.get(f'/api/v1/interviews/sessions/{session_id}')
    r.raise_for_status()
    return r.json()


def start_session(candidate_id: str, job_id: str = JOB_ID) -> dict:
    """Crea una sesión nueva. Si ya existe, la retoma desde donde quedó."""
    r = client.post('/api/v1/interviews/sessions', json={
        'job_id': job_id,
        'candidate_id': candidate_id,
    })
    if r.status_code == 409:
        detail = r.json()['detail']
        session_id = detail['session_id']
        state = get_session_state(session_id)
        idx = state['current_question_index']
        if state['status'] == 'completed':
            print(f'  ⚠ Sesión ya existe y está completada: {session_id}')
        else:
            print(f'  ⚠ Sesión ya existe: {session_id} — retomando desde pregunta {idx + 1}/8')
        return {'session_id': session_id, 'already_existed': True, 'start_index': idx, '_status': state['status']}
    r.raise_for_status()
    data = r.json()
    print(f'  ✓ Sesión creada: {data["session_id"]}')
    return {**data, 'already_existed': False, 'start_index': 0, '_status': 'active'}


def send_answer(session_id: str, answer: str) -> dict:
    r = client.post(f'/api/v1/interviews/sessions/{session_id}/message', json={'content': answer})
    r.raise_for_status()
    return r.json()


print('✓ Helpers definidos')

✓ Helpers definidos


In [5]:
# ── Paso 3: correr la entrevista completa para cada candidato ─────────────────

all_results = []

for cand in CANDIDATES:
    print(f"\n{'='*60}")
    print(f"Candidato: {cand['name']}")
    print(f"Escenario: {cand['scenario']}")
    print(f"{'='*60}")

    scenario = load_scenario(cand['scenario'])

    # Crear sesión (o retomar la existente con su current_question_index)
    session_data = start_session(cand['candidate_id'])
    session_id = session_data['session_id']
    start_index = session_data.get('start_index', 0)

    # Si la sesión ya estaba completada, no hay nada que enviar
    if session_data.get('_status') == 'completed':
        print(f'  ⚠ Sesión ya completada. Recupera el resultado desde la DB.')
        print(f'    Para reintentar borra la sesión o usa otro candidate_id.')
        all_results.append({
            'candidate': cand['name'],
            'scenario': cand['scenario'],
            'session_id': session_id,
            'evaluation_result': None,
            'status': 'already_completed',
        })
        continue

    if start_index > 0:
        print(f'  ⚠ Saltando D1–D{start_index} (ya respondidas en sesión anterior)')

    # Enviar respuestas del dataset desde start_index
    evaluation_result = None
    answers = scenario['answers']

    for i, answer_data in enumerate(answers[start_index:], start=start_index):
        dim_id = answer_data['dimension_id']
        answer_text = answer_data['answer']

        print(f'  Enviando respuesta {i+1}/8 [{dim_id}]...', end=' ', flush=True)
        t0 = time.monotonic()

        try:
            response = send_answer(session_id, answer_text)
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 410:
                elapsed = time.monotonic() - t0
                print(f'✗ Sesión completada en el servidor (410) ({elapsed:.1f}s)')
                print(f'    La evaluación fue disparada por respuesta anterior.')
                print(f'    Revisa la DB para ver el resultado completo.')
                break
            raise

        elapsed = time.monotonic() - t0
        if response['evaluation_result']:
            evaluation_result = response['evaluation_result']
            print(f'✓ Evaluación completa ({elapsed:.1f}s)')
        else:
            next_q = response['next_question']
            print(f'✓ siguiente: {next_q["dimension_id"]} ({elapsed:.1f}s)')

    all_results.append({
        'candidate': cand['name'],
        'scenario': cand['scenario'],
        'session_id': session_id,
        'evaluation_result': evaluation_result,
        'status': 'completed' if evaluation_result else 'incomplete',
        'expected': scenario['expected_result'],
    })

completed_count = sum(1 for r in all_results if r['evaluation_result'])
print(f"\n✓ Evaluaciones obtenidas: {completed_count}/{len(CANDIDATES)}")


Candidato: Sofía Delgado
Escenario: ganador_claro
  ✓ Sesión creada: c7eeccea-9006-40a6-9dfe-ff04c7700dae
  Enviando respuesta 1/8 [D1]... ✓ siguiente: D2 (0.1s)
  Enviando respuesta 2/8 [D2]... ✓ siguiente: D3 (0.0s)
  Enviando respuesta 3/8 [D3]... ✓ siguiente: D4 (0.0s)
  Enviando respuesta 4/8 [D4]... ✓ siguiente: D5 (0.1s)
  Enviando respuesta 5/8 [D5]... ✓ siguiente: D6 (0.0s)
  Enviando respuesta 6/8 [D6]... ✓ siguiente: D7 (0.0s)
  Enviando respuesta 7/8 [D7]... ✓ siguiente: D8 (0.0s)
  Enviando respuesta 8/8 [D8]... ✓ Evaluación completa (15.2s)

Candidato: Carlos Rivas
Escenario: buen_candidato
  ✓ Sesión creada: c85c9053-f424-4f21-83aa-3e0a6ce14a27
  Enviando respuesta 1/8 [D1]... ✓ siguiente: D2 (0.0s)
  Enviando respuesta 2/8 [D2]... ✓ siguiente: D3 (0.0s)
  Enviando respuesta 3/8 [D3]... ✓ siguiente: D4 (0.0s)
  Enviando respuesta 4/8 [D4]... ✓ siguiente: D5 (0.0s)
  Enviando respuesta 5/8 [D5]... ✓ siguiente: D6 (0.0s)
  Enviando respuesta 6/8 [D6]... ✓ siguiente: D7 (0

In [6]:
# ── Paso 4: comparar scores reales vs esperados ───────────────────────────────

SCORE_EMOJI = {5: '🟢', 4: '🟡', 3: '🟠', 2: '🔴', 1: '⛔'}

for result in all_results:
    print(f"\n{'='*60}")
    print(f"  {result['candidate']} — {result['scenario']}")
    print(f"  Sesión: {result['session_id']}")

    if result['status'] == 'already_completed':
        print('  ⚠ Sesión ya estaba completada antes de este notebook.')
        print('  Para volver a probar borra la sesión de la DB o usa otro candidate_id.')
        continue

    er = result['evaluation_result']
    if not er:
        print('  ✗ No se obtuvo resultado de evaluación')
        continue

    real_score = Decimal(er['weighted_score'])
    expected_score = Decimal(result['expected']['weighted_score'])
    diff = real_score - expected_score

    print(f"\n  Score real (OpenAI):    {real_score:.2f} / 5.00")
    print(f"  Score esperado (mock):  {expected_score:.2f} / 5.00")
    print(f"  Diferencia:             {diff:+.2f}")
    print(f"  Resultado:              {er['resultado']}")

    print(f"\n  Scores por dimensión:")
    scenario = load_scenario(result['scenario'])
    expected_dim_scores = {a['dimension_id']: a['simulated_ai_score'] for a in scenario['answers']}

    for ds in er['dimension_scores']:
        dim = ds['dimension_id']
        real = int(ds['score'].split('.')[0]) if '.' in ds['score'] else int(ds['score'])
        expected = expected_dim_scores.get(dim, '?')
        match = '✓' if real == expected else '≠'
        emoji = SCORE_EMOJI.get(real, '?')
        print(f"    {emoji} {dim} → real: {real}/5  esperado: {expected}/5  {match}  | {ds['justificacion'][:60]}...")


  Sofía Delgado — ganador_claro
  Sesión: c7eeccea-9006-40a6-9dfe-ff04c7700dae

  Score real (OpenAI):    4.68 / 5.00
  Score esperado (mock):  4.84 / 5.00
  Diferencia:             -0.16
  Resultado:              APTO

  Scores por dimensión:
    🟠 D1 → real: 3/5  esperado: 5/5  ≠  | El candidato tiene experiencia con la integración de WhatsAp...
    🟢 D2 → real: 5/5  esperado: 5/5  ✓  | Cumple con todos los requisitos del nivel 5, mostrando exper...
    🟢 D3 → real: 5/5  esperado: 5/5  ✓  | El candidato ha liderado un proyecto completo en solitario, ...
    🟢 D4 → real: 5/5  esperado: 4/5  ≠  | Demuestra explícitamente uso de herramientas existentes para...
    🟢 D5 → real: 5/5  esperado: 5/5  ✓  | El candidato presenta un historial documentado de entrega en...
    🟢 D6 → real: 5/5  esperado: 5/5  ✓  | El candidato demuestra una comunicación efectiva con no técn...
    🟢 D7 → real: 5/5  esperado: 5/5  ✓  | El candidato tiene experiencia directa en proyectos de salud...
    🟢 D8 → re

In [7]:
# ── Paso 5: ranking real vs ranking esperado ──────────────────────────────────

completed = [r for r in all_results if r['status'] == 'completed' and r['evaluation_result']]

if not completed:
    print('No hay resultados completos para mostrar el ranking.')
else:
    ranked = sorted(completed, key=lambda r: Decimal(r['evaluation_result']['weighted_score']), reverse=True)

    print(f"\n{'='*65}")
    print("RANKING REAL (scores evaluados por OpenAI gpt-4o-mini)")
    print(f"{'='*65}")
    print(f"{'Pos':<4} {'Candidato':<22} {'Score real':>11} {'Score esp.':>11} {'Dif.':>6} {'OK?'}")
    print('-' * 60)

    MEDALS = ['🥇', '🥈', '🥉', '  4']
    for i, r in enumerate(ranked):
        real = Decimal(r['evaluation_result']['weighted_score'])
        expected = Decimal(r['expected']['weighted_score'])
        diff = real - expected
        ok = '✓' if abs(diff) <= Decimal('1.0') else '⚠ diff grande'
        medal = MEDALS[i] if i < len(MEDALS) else f'  {i+1}'
        print(f"{medal}   {r['candidate']:<22} {float(real):>10.2f}  {float(expected):>10.2f}  {float(diff):>+5.2f}  {ok}")

    print(f"\n{'='*65}")

    # ¿El ganador real coincide con el ganador esperado?
    if ranked[0]['scenario'] == 'ganador_claro':
        print(f"\n🏆 GANADOR CORRECTO: {ranked[0]['candidate']}")
        print(f"   Score real: {ranked[0]['evaluation_result']['weighted_score']} / 5.00")
        print(f"   La IA confirmó al candidato ganador esperado ✓")
    else:
        print(f"\n⚠ GANADOR DIFERENTE al esperado: {ranked[0]['candidate']}")
        print(f"   El esperado era 'ganador_claro'. Las respuestas fueron evaluadas diferente por la IA.")
        print(f"   Esto es normal — la IA tiene criterio propio dentro de la rúbrica.")


RANKING REAL (scores evaluados por OpenAI gpt-4o-mini)
Pos  Candidato               Score real  Score esp.   Dif. OK?
------------------------------------------------------------
🥇   Sofía Delgado                4.68        4.84  -0.16  ✓
🥈   Carlos Rivas                 3.74        4.74  -1.00  ✓
🥉   Elena Martínez               3.11        3.58  -0.47  ✓


🏆 GANADOR CORRECTO: Sofía Delgado
   Score real: 4.68 / 5.00
   La IA confirmó al candidato ganador esperado ✓


In [ ]:
# ── Paso 6: visualización ─────────────────────────────────────────────────────

try:
    import matplotlib.pyplot as plt
    import numpy as np

    if not completed:
        print('Sin datos para graficar.')
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        fig.suptitle('NovaHiring — Evaluación real (gpt-4o-mini)', fontsize=13, fontweight='bold')

        # Grafico 1: score real vs esperado por candidato
        names = [r['candidate'].split()[0] for r in completed]
        reals = [float(Decimal(r['evaluation_result']['weighted_score'])) for r in completed]
        expected_vals = [float(Decimal(r['expected']['weighted_score'])) for r in completed]

        x = np.arange(len(names))
        w = 0.35
        ax1.bar(x - w/2, reals, w, label='Score real (OpenAI)', color='#3498db', alpha=0.85)
        ax1.bar(x + w/2, expected_vals, w, label='Score esperado (mock)', color='#95a5a6', alpha=0.85)
        ax1.set_xticks(x)
        ax1.set_xticklabels(names)
        ax1.set_ylim(0, 5.5)
        ax1.axhline(y=3, color='red', linestyle='--', alpha=0.4, label='Mínimo aceptable')
        ax1.set_ylabel('Score (1–5)')
        ax1.set_title('Score real vs esperado por candidato')
        ax1.legend()
        for i, (r, e) in enumerate(zip(reals, expected_vals)):
            ax1.text(i - w/2, r + 0.05, f'{r:.2f}', ha='center', fontsize=9, fontweight='bold', color='#2980b9')
            ax1.text(i + w/2, e + 0.05, f'{e:.2f}', ha='center', fontsize=9, color='#7f8c8d')

        # Grafico 2: scores por dimensión del ganador
        winner = sorted(completed, key=lambda r: Decimal(r['evaluation_result']['weighted_score']), reverse=True)[0]
        scenario = load_scenario(winner['scenario'])
        dim_ids = [a['dimension_id'] for a in scenario['answers']]
        real_dim_scores = {ds['dimension_id']: float(ds['score']) for ds in winner['evaluation_result']['dimension_scores']}
        mock_dim_scores = {a['dimension_id']: a['simulated_ai_score'] for a in scenario['answers']}
        pesos = {'D1': 3, 'D2': 3, 'D3': 3, 'D4': 3, 'D5': 2, 'D6': 2, 'D7': 2, 'D8': 1}

        dim_labels = [f"{d}\n(×{pesos[d]})" for d in dim_ids]
        y_real = [real_dim_scores.get(d, 0) for d in dim_ids]
        y_mock = [mock_dim_scores.get(d, 0) for d in dim_ids]

        x2 = np.arange(len(dim_ids))
        ax2.bar(x2 - w/2, y_real, w, label='Real (OpenAI)', color='#2ecc71', alpha=0.85)
        ax2.bar(x2 + w/2, y_mock, w, label='Mock esperado', color='#95a5a6', alpha=0.85)
        ax2.set_xticks(x2)
        ax2.set_xticklabels(dim_labels, fontsize=8)
        ax2.set_ylim(0, 5.5)
        ax2.axhline(y=3, color='red', linestyle='--', alpha=0.4)
        ax2.set_ylabel('Score (1–5)')
        ax2.set_title(f'Scores por dimensión — {winner["candidate"].split()[0]} (ganador)')
        ax2.legend()

        plt.tight_layout()
        plt.savefig(ROOT / 'notebooks' / 'e2e_scores.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✓ Gráfico guardado en notebooks/e2e_scores.png')

except ImportError:
    print('matplotlib no instalado → uv add matplotlib')

In [8]:
# ── Paso 7: verificar que los datos están en la DB ────────────────────────────
# (Opcional — requiere psycopg2 o asyncpg directo)

print('Para verificar en la base de datos directamente:')
print()
print('  docker exec -it nova-postgres-1 psql -U nova -d nova_hiring')
print()
print('  -- Todas las evaluaciones de entrevista:')
print("  SELECT e.id, c.nombre, e.weighted_score, e.resultado, e.created_at")
print("  FROM evaluations e")
print("  JOIN candidates c ON c.id = e.candidate_id")
print("  ORDER BY e.created_at DESC LIMIT 10;")
print()
print('  -- Scores por dimensión de la última evaluación:')
print("  SELECT dimension_id, raw_score, peso, justificacion")
print("  FROM dimension_scores")
print("  WHERE evaluation_id = (SELECT id FROM evaluations ORDER BY created_at DESC LIMIT 1);")
print()
print('  -- Logs de llamadas a OpenAI:')
print("  SELECT prompt_name, model, input_tokens, output_tokens, latency_ms, validation_passed")
print("  FROM ai_call_logs")
print("  ORDER BY created_at DESC LIMIT 8;")

Para verificar en la base de datos directamente:

  docker exec -it nova-postgres-1 psql -U nova -d nova_hiring

  -- Todas las evaluaciones de entrevista:
  SELECT e.id, c.nombre, e.weighted_score, e.resultado, e.created_at
  FROM evaluations e
  JOIN candidates c ON c.id = e.candidate_id
  ORDER BY e.created_at DESC LIMIT 10;

  -- Scores por dimensión de la última evaluación:
  SELECT dimension_id, raw_score, peso, justificacion
  FROM dimension_scores
  WHERE evaluation_id = (SELECT id FROM evaluations ORDER BY created_at DESC LIMIT 1);

  -- Logs de llamadas a OpenAI:
  SELECT prompt_name, model, input_tokens, output_tokens, latency_ms, validation_passed
  FROM ai_call_logs
  ORDER BY created_at DESC LIMIT 8;


## ¿Qué acaba de pasar?

```
Notebook                          Servidor FastAPI              OpenAI
─────────                         ───────────────              ──────
POST /sessions           ──────►  Crea ChatSession en DB
                                  Guarda context_summary JSON
                         ◄──────  session_id + mensaje bienvenida

POST /message (×8)       ──────►  Guarda respuesta en messages
                                  [En la 8ª respuesta:]
                                  Carga rúbricas del discovery_json
                                  Para cada D1..D8:              
                                    AIClient.call() ──────────► gpt-4o-mini
                                                    ◄────────── {score, justif, evidencia}
                                  Scorer.calculate()  (Python puro)
                                  INSERT evaluations
                                  INSERT dimension_scores ×8
                                  ChatSession.status = 'completed'
                         ◄──────  evaluation_result con score final
```

**La IA solo hizo 8 cosas:** puntuar 8 respuestas de texto. El resto fue Python.